In [20]:
import pandas as pd
from joblib import load
from sklearn.metrics import f1_score,confusion_matrix
import numpy as np

In [21]:
oxide = pd.read_excel(r"D:/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/oxide_data.xlsx", sheet_name="Sheet1",index_col=0,header=0)

def normalize(data,n=100):
    s = n/data.sum(axis=1)
    data_formatted = data.mul(s,axis=0)
    return(data_formatted)
    
def wt_to_mol(data, oxide = oxide):
    data[data<=2] = 0
    if 'Total' in data.columns:
        data.pop('Total')
    oxlist1 = data.columns
    oxide = oxide.T[oxlist1].iloc[0, :]
    data = normalize(data,100)
    data_f = data.div(oxide,axis=1)
    data_f = normalize(data_f)
    return(data_f.round(2))

# file = r"D:/Research/Mineral identifier ann IISER Mohali/new random comp generator/GEOROC data/processed minerals/molar tables/new data/compiled9_2.xlsx"
# file_test = r"D:/Research/Mineral identifier ann IISER Mohali/new random comp generator/External test data/Mineral data combined_RSandSS_mol1.xlsx"
# file_test = r"D:\Research\Mineral identifier ann IISER Mohali\new random comp generator\External test data\Mineral data combined_DHZ,PSandSS.xlsx"
# file_test = r"D:\Research\Mineral identifier ann IISER Mohali\new random comp generator\External test data\test_data_deerhowiezussmann.xlsx"
file_test = r"D:\Research\Mineral identifier ann IISER Mohali\ms and reference papers\MinNet ver 4 for submitted to AmMin\Revision 1\AmMin revision 1\Sir modified files\External dataset 2\Processed\name shortened\Combined_PantSir.xlsx"

# data = pd.read_excel(file,header=0,index_col=0)
data_test = pd.read_excel(file_test,header=0,index_col=0)
m = len(data_test['Mineral'].unique())
acc_table = np.zeros((m+1,5))

data_test

,SiO2,MgO,FeO,TiO2,Al2O3,MnO,CaO,Na2O,K2O,Cr2O3,...,Er2O3,Yb2O3,V2O3,CoO,ZnO,ZrO2,CuO,Total,Mineral,Source
1,40.79,48.85,10.16,0.000,0.00,0.000,0.000,0.000,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.800,Ol,Batanova et al 2019
2,40.47,48.68,10.20,0.000,0.00,0.000,0.000,0.000,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.350,Ol,Batanova et al 2019
3,40.96,48.83,10.16,0.000,0.00,0.000,0.000,0.000,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.950,Ol,Batanova et al 2019
4,55.77,6.37,0.02,0.010,23.62,0.000,0.000,0.100,11.23,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,97.120,Ms,Biino and Gröning - 1998
5,55.65,6.29,0.01,0.000,23.67,0.000,0.000,0.080,11.17,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,96.870,Ms,Biino and Gröning - 1998
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1422,48.50,0.00,4.40,0.027,26.50,1.327,0.008,0.324,10.20,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,91.286,Ms,Xu et al 2023
1423,47.40,1.26,4.07,0.363,32.50,0.221,0.017,0.617,9.57,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,96.018,Ms,Xu et al 2023
1424,45.60,0.52,4.76,0.357,32.00,0.694,0.011,0.699,9.90,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,94.541,Ms,Xu et al 2023
1425,47.80,0.00,0.93,0.067,36.60,0.349,0.002,0.327,10.31,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,96.385,Ms,Xu et al 2023


In [22]:
if 'Mineral' in data_test.columns:
    minerals = data_test.pop("Mineral")
if 'Source' in data_test.columns:
    source = data_test.pop("Source")

# if 'Fe2O3' in data_test.columns:
#     data_test['FeO'] = data_test['FeO'] + (0.8998 * data_test['Fe2O3'])
#     data_test.pop('Fe2O3')

if 'F' in data_test.columns:
    data_test.pop('F')
if 'Cl' in data_test.columns:
    data_test.pop('Cl')

data_test = normalize(data_test)

metrics = 'weighted'
if 'Total' not in data_test.columns:
    data_test['Total'] = data_test.iloc[:,:-2].sum(axis=1)
data_test.loc[minerals=='Cb','CO2'] = data_test['Total'] - data_test[['FeO','MnO','MgO','CaO']].sum(axis=1)

In [23]:
data_test.fillna(0,inplace=True)
data_test_original = data_test.copy()
data_test = data_test.round(2)

In [24]:
data_test = wt_to_mol(data_test,oxide)
if 'SrO' in data_test.columns:
    data_test['CaO'] = data_test['CaO'] + data_test['SrO']
    data_test.pop('SrO')
if 'BaO' in data_test.columns:
    data_test['CaO'] = data_test['CaO'] + data_test['BaO']
    data_test.pop('BaO')
if 'Cr2O3' in data_test.columns:
    data_test['Al2O3'] = data_test['Al2O3'] + data_test['Cr2O3']
    data_test.pop('Cr2O3')
if 'ZnO' in data_test.columns:
    data_test['FeO'] = data_test['FeO'] + data_test['ZnO']
    data_test.pop('ZnO')
if 'CoO' in data_test.columns:
    data_test['FeO'] = data_test['FeO'] + data_test['CoO']
    data_test.pop('CoO')

In [25]:
data_test

,SiO2,MgO,FeO,TiO2,Al2O3,MnO,CaO,Na2O,K2O,P2O5,...,Nb2O5,Sm2O3,Gd2O3,Dy2O3,Er2O3,Yb2O3,V2O3,ZrO2,CuO,CO2
1,33.411789,59.630022,6.958189,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,33.292916,59.695151,7.011933,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,33.503375,59.550063,6.946562,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,64.585902,10.999946,0.0,0.0,16.12009,0.0,0.0,0.0,8.294062,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,64.616749,10.900745,0.0,0.0,16.201723,0.0,0.0,0.0,8.280783,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1422,65.271151,0.0,4.953017,0.0,21.013092,0.0,0.0,0.0,8.76274,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1423,62.322679,0.0,4.477096,0.0,25.178547,0.0,0.0,0.0,8.021678,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1424,61.000412,0.0,5.329858,0.0,25.216521,0.0,0.0,0.0,8.45321,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1425,62.938971,0.0,0.0,0.0,28.400256,0.0,0.0,0.0,8.660773,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Raw Input (C1)

In [26]:
model = load("SVM_C1.mdl")
scaler = load("Scaler_C1.scl")
labeler = load("Labeler_C1.lbl")

data_test1 = data_test.copy()
# data_test1['M'] = data_test1[["FeO","MnO","MgO"]].sum(axis=1)
data_test1 = data_test1[['SiO2','TiO2','Al2O3','FeO','MnO','MgO','CaO','Na2O','K2O','P2O5','CO2']]

# m = data_test1[["FeO","MnO","MgO"]]sum(axis=1)

data_test_scaled = scaler.transform(data_test1)
test_target = labeler.transform(minerals)
pred_label = model.predict(data_test_scaled)
a = round(f1_score(test_target, pred_label, average=metrics,zero_division=np.nan),4)*100
acc_table[0,0] = a
print("Overall: "+str(a))

labels = labeler.transform(minerals.unique())
labels1 = labeler.inverse_transform(labels)

for n,i in enumerate(labels1):
    data_test2 = scaler.transform(data_test1.loc[minerals==i,:])
    minerals2 = labeler.transform(minerals.loc[minerals == i])
    pred_label2 = model.predict(data_test2)
    a = round(f1_score(minerals2, pred_label2, average=metrics,zero_division=0),4)*100
    acc_table[n+1,0] = a
    print(labels1[n]+": "+str(round(a,2)))

data_mismatch = data_test.loc[test_target != pred_label,:]
minerals_mismatch = minerals.loc[test_target != pred_label]
data_mismatch = pd.concat([data_mismatch,minerals_mismatch],axis=1)
data_mismatch

C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.5.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unp

Overall: 98.32
Ol: 100.0
Ms: 100.0
Grt: 94.44
Amp: 98.23
Px: 92.65
Spl: 100.0
Fsp: 100.0
Bt: 100.0
Ttn: 100.0
Ap: 100.0


,SiO2,MgO,FeO,TiO2,Al2O3,MnO,CaO,Na2O,K2O,P2O5,...,Sm2O3,Gd2O3,Dy2O3,Er2O3,Yb2O3,V2O3,ZrO2,CuO,CO2,Mineral
66,51.715342,14.220639,14.846501,0.0,9.974916,0.0,9.242601,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
67,47.588864,17.947578,9.627537,0.0,6.377902,0.0,18.458119,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
68,50.028702,22.641913,7.416407,0.0,7.965032,0.0,11.947946,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
70,48.476941,22.723663,7.254025,0.0,3.710148,0.0,17.835223,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
867,42.062995,20.159063,10.880062,3.582479,9.353467,0.0,13.961934,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
869,59.298273,0.0,27.112152,0.0,0.0,0.0,0.0,13.589575,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
871,44.359964,21.170997,12.041714,0.0,9.610435,0.0,12.816889,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
872,42.641425,27.606305,3.750115,0.0,11.010401,0.0,14.991753,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
922,53.505052,26.337171,5.763729,0.0,0.0,0.0,14.394048,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
923,57.856076,13.420834,0.0,0.0,7.848197,0.0,14.689607,6.185286,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px


# C2

In [27]:
model = load("SVM_C2.mdl")
scaler = load("Scaler_C2.scl")
labeler = load("Labeler_C2.lbl")

data_test1 = data_test.copy()
data_test1['M'] = data_test1[["FeO","MnO","MgO"]].sum(axis=1)
data_test1 = data_test1[['SiO2','TiO2','Al2O3','M','CaO','Na2O','K2O','P2O5','CO2']]

# m = data_test1[["FeO","MnO","MgO"]]sum(axis=1)

data_test_scaled = scaler.transform(data_test1)
test_target = labeler.transform(minerals)
pred_label = model.predict(data_test_scaled)
a = round(f1_score(test_target, pred_label, average=metrics,zero_division=0),4)*100
acc_table[0,1] = a
print("Overall: "+str(a))

labels = labeler.transform(minerals.unique())
labels1 = labeler.inverse_transform(labels)

for n,i in enumerate(labels1):
    data_test2 = scaler.transform(data_test1.loc[minerals==i,:])
    minerals2 = labeler.transform(minerals.loc[minerals == i])
    pred_label2 = model.predict(data_test2)
    a = round(f1_score(minerals2, pred_label2, average=metrics,zero_division=0),4)*100
    acc_table[n+1,1] = a
    print(labels1[n]+": "+str(round(a,2)))

data_mismatch = data_test.loc[test_target != pred_label,:]
minerals_mismatch = minerals.loc[test_target != pred_label]
data_mismatch = pd.concat([data_mismatch,minerals_mismatch],axis=1)
# data_mismatch['Pred'] = labeler.inverse_transform(pred_label)
data_mismatch

C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.5.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unp

Overall: 98.37
Ol: 100.0
Ms: 100.0
Grt: 94.44
Amp: 98.23
Px: 93.43
Spl: 100.0
Fsp: 100.0
Bt: 100.0
Ttn: 100.0
Ap: 100.0


,SiO2,MgO,FeO,TiO2,Al2O3,MnO,CaO,Na2O,K2O,P2O5,...,Sm2O3,Gd2O3,Dy2O3,Er2O3,Yb2O3,V2O3,ZrO2,CuO,CO2,Mineral
66,51.715342,14.220639,14.846501,0.0,9.974916,0.0,9.242601,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
67,47.588864,17.947578,9.627537,0.0,6.377902,0.0,18.458119,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
68,50.028702,22.641913,7.416407,0.0,7.965032,0.0,11.947946,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
70,48.476941,22.723663,7.254025,0.0,3.710148,0.0,17.835223,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
867,42.062995,20.159063,10.880062,3.582479,9.353467,0.0,13.961934,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
869,59.298273,0.0,27.112152,0.0,0.0,0.0,0.0,13.589575,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
871,44.359964,21.170997,12.041714,0.0,9.610435,0.0,12.816889,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
872,42.641425,27.606305,3.750115,0.0,11.010401,0.0,14.991753,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
922,53.505052,26.337171,5.763729,0.0,0.0,0.0,14.394048,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
923,57.856076,13.420834,0.0,0.0,7.848197,0.0,14.689607,6.185286,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px


# C3

In [28]:
model = load("SVM_C3.mdl")
scaler = load("Scaler_C3.scl")
labeler = load("Labeler_C3.lbl")

data_test1 = data_test.copy()
data_test1['M'] = data_test1[["FeO","MnO","MgO"]].sum(axis=1)
data_test1['A'] = data_test1[["Na2O","K2O"]].sum(axis=1)
data_test1 = data_test1[['SiO2','TiO2','Al2O3','M','CaO','A','P2O5','CO2']]

# m = data_test1[["FeO","MnO","MgO"]]sum(axis=1)

data_test_scaled = scaler.transform(data_test1)
test_target = labeler.transform(minerals)
pred_label = model.predict(data_test_scaled)
a = round(f1_score(test_target, pred_label, average=metrics,zero_division=0),4)*100
acc_table[0,2] = a
print("Overall: "+str(a))

labels = labeler.transform(minerals.unique())
labels1 = labeler.inverse_transform(labels)

for n,i in enumerate(labels1):
    data_test2 = scaler.transform(data_test1.loc[minerals==i,:])
    minerals2 = labeler.transform(minerals.loc[minerals == i])
    pred_label2 = model.predict(data_test2)
    a = round(f1_score(minerals2, pred_label2, average=metrics,zero_division=0),4)*100
    acc_table[n+1,2] = a
    print(labels1[n]+": "+str(round(a,2)))

data_mismatch = data_test.loc[test_target != pred_label,:]
minerals_mismatch = minerals.loc[test_target != pred_label]
data_mismatch = pd.concat([data_mismatch,minerals_mismatch],axis=1)
data_mismatch

C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.5.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unp

Overall: 98.44000000000001
Ol: 100.0
Ms: 100.0
Grt: 94.44
Amp: 98.68
Px: 93.43
Spl: 100.0
Fsp: 100.0
Bt: 100.0
Ttn: 100.0
Ap: 100.0


,SiO2,MgO,FeO,TiO2,Al2O3,MnO,CaO,Na2O,K2O,P2O5,...,Sm2O3,Gd2O3,Dy2O3,Er2O3,Yb2O3,V2O3,ZrO2,CuO,CO2,Mineral
66,51.715342,14.220639,14.846501,0.0,9.974916,0.0,9.242601,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
67,47.588864,17.947578,9.627537,0.0,6.377902,0.0,18.458119,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
68,50.028702,22.641913,7.416407,0.0,7.965032,0.0,11.947946,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
70,48.476941,22.723663,7.254025,0.0,3.710148,0.0,17.835223,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
867,42.062995,20.159063,10.880062,3.582479,9.353467,0.0,13.961934,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
871,44.359964,21.170997,12.041714,0.0,9.610435,0.0,12.816889,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
872,42.641425,27.606305,3.750115,0.0,11.010401,0.0,14.991753,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
922,53.505052,26.337171,5.763729,0.0,0.0,0.0,14.394048,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
923,57.856076,13.420834,0.0,0.0,7.848197,0.0,14.689607,6.185286,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
924,52.498441,28.750389,4.337982,0.0,0.0,0.0,14.413188,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px


# C4

In [29]:
model = load("SVM_C4.mdl")
scaler = load("Scaler_C4.scl")
labeler = load("Labeler_C4.lbl")

data_test1 = data_test.copy()
data_test1['M'] = data_test1[["FeO","MnO","MgO","CaO"]].sum(axis=1)
data_test1['A'] = data_test1[["CaO","Na2O","K2O"]].sum(axis=1)
data_test1 = data_test1[['SiO2','TiO2','Al2O3','M','A','P2O5','CO2']]

# m = data_test1[["FeO","MnO","MgO"]]sum(axis=1)

data_test_scaled = scaler.transform(data_test1)
test_target = labeler.transform(minerals)
pred_label = model.predict(data_test_scaled)
a = round(f1_score(test_target, pred_label, average=metrics,zero_division=0),4)*100
acc_table[0,3] = a
print("Overall: "+str(a))

labels = labeler.transform(minerals.unique())
labels1 = labeler.inverse_transform(labels)

for n,i in enumerate(labels1):
    data_test2 = scaler.transform(data_test1.loc[minerals==i,:])
    minerals2 = labeler.transform(minerals.loc[minerals == i])
    pred_label2 = model.predict(data_test2)
    a = round(f1_score(minerals2, pred_label2, average=metrics,zero_division=0),4)*100
    acc_table[n+1,3] = a
    print(labels1[n]+": "+str(round(a,2)))

data_mismatch = data_test.loc[test_target != pred_label,:]
minerals_mismatch = minerals.loc[test_target != pred_label]
data_mismatch = pd.concat([data_mismatch,minerals_mismatch],axis=1)
data_mismatch

C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.5.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unp

Overall: 97.58
Ol: 100.0
Ms: 97.37
Grt: 90.17
Amp: 98.23
Px: 93.43
Spl: 100.0
Fsp: 99.56
Bt: 99.82
Ttn: 100.0
Ap: 100.0


,SiO2,MgO,FeO,TiO2,Al2O3,MnO,CaO,Na2O,K2O,P2O5,...,Sm2O3,Gd2O3,Dy2O3,Er2O3,Yb2O3,V2O3,ZrO2,CuO,CO2,Mineral
4,64.585902,10.999946,0.0,0.0,16.12009,0.0,0.0,0.0,8.294062,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Ms
5,64.616749,10.900745,0.0,0.0,16.201723,0.0,0.0,0.0,8.280783,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Ms
66,51.715342,14.220639,14.846501,0.0,9.974916,0.0,9.242601,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
67,47.588864,17.947578,9.627537,0.0,6.377902,0.0,18.458119,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
68,50.028702,22.641913,7.416407,0.0,7.965032,0.0,11.947946,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
70,48.476941,22.723663,7.254025,0.0,3.710148,0.0,17.835223,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
834,51.056807,6.66331,0.0,0.0,19.669431,0.0,22.610452,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Fsp
863,36.088923,32.184496,0.0,0.0,9.062078,16.707944,0.0,0.0,5.956559,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Bt
865,44.518963,20.642069,9.715551,4.061939,9.119585,0.0,11.941892,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
867,42.062995,20.159063,10.880062,3.582479,9.353467,0.0,13.961934,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp


# C5

In [30]:
model = load("SVM_C5_5_components.mdl")
scaler = load("Scaler_C5_5_components.scl")
labeler = load("Labeler_C5_5_components.lbl")
pc = load("PCA_C5_5_components.pc")

# model = load("SVM_C5_6_components.mdl")
# scaler = load("Scaler_C5_6_components.scl")
# labeler = load("Labeler_C5_6_components.lbl")
# pc = load("PCA_C5_6_components.pc")

data_test1 = data_test.copy()
data_test1 = data_test1[['SiO2','TiO2','Al2O3','FeO','MnO','MgO','CaO','Na2O','K2O','P2O5','CO2']]
# m = data_test1[["FeO","MnO","MgO"]]sum(axis=1)

data_test_scaled = scaler.transform(data_test1)
data_test_scaled = pc.transform(data_test_scaled)
test_target = labeler.transform(minerals)
pred_label = model.predict(data_test_scaled)
a = round(f1_score(test_target, pred_label, average=metrics,zero_division=0),4)*100
acc_table[0,4] = a
print("Overall: "+str(a))

labels = labeler.transform(minerals.unique())
labels1 = labeler.inverse_transform(labels)

for n,i in enumerate(labels1):
    data_test2 = scaler.transform(data_test1.loc[minerals==i,:])
    data_test2 = pc.transform(data_test2)
    minerals2 = labeler.transform(minerals.loc[minerals == i])
    pred_label2 = model.predict(data_test2)
    a = round(f1_score(minerals2, pred_label2, average=metrics,zero_division=0),4)*100
    acc_table[n+1,4] = a
    print(labels1[n]+": "+str(round(a,2)))

data_mismatch = data_test.loc[test_target != pred_label,:]
minerals_mismatch = minerals.loc[test_target != pred_label]
data_mismatch = pd.concat([data_mismatch,minerals_mismatch],axis=1)
data_mismatch

C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.5.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.0 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\naik3\Documents\python_geology\granite_mineralization\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unp

Overall: 96.69
Ol: 100.0
Ms: 100.0
Grt: 80.5
Amp: 98.68
Px: 93.43
Spl: 100.0
Fsp: 99.56
Bt: 99.82
Ttn: 100.0
Ap: 100.0


,SiO2,MgO,FeO,TiO2,Al2O3,MnO,CaO,Na2O,K2O,P2O5,...,Sm2O3,Gd2O3,Dy2O3,Er2O3,Yb2O3,V2O3,ZrO2,CuO,CO2,Mineral
66,51.715342,14.220639,14.846501,0.0,9.974916,0.0,9.242601,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
67,47.588864,17.947578,9.627537,0.0,6.377902,0.0,18.458119,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
68,50.028702,22.641913,7.416407,0.0,7.965032,0.0,11.947946,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
70,48.476941,22.723663,7.254025,0.0,3.710148,0.0,17.835223,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px
834,51.056807,6.66331,0.0,0.0,19.669431,0.0,22.610452,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Fsp
863,36.088923,32.184496,0.0,0.0,9.062078,16.707944,0.0,0.0,5.956559,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Bt
867,42.062995,20.159063,10.880062,3.582479,9.353467,0.0,13.961934,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
869,59.298273,0.0,27.112152,0.0,0.0,0.0,0.0,13.589575,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
872,42.641425,27.606305,3.750115,0.0,11.010401,0.0,14.991753,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Amp
922,53.505052,26.337171,5.763729,0.0,0.0,0.0,14.394048,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Px


In [12]:
l=['Overall']
l.extend(list(minerals.unique()))
acc_table = pd.DataFrame(acc_table,
                          columns=["C1", "C2", "C3", "C4", "C5"],index=l)
acc_table.to_excel("F1-score Accuracy report for SVM_test.xlsx")


In [13]:
l=['Overall']
l.extend(list(minerals.unique()))
l

['Overall',
 'St',
 'Ol',
 'Grt',
 'Crd',
 'Ms',
 'Bt',
 'Px',
 'Fsp',
 'Spl',
 'Qz',
 'Rt',
 'Ilm',
 'Als',
 'Amp',
 'Cb',
 'Ap',
 'Mag/Hem',
 'Ttn']

In [14]:
for n,i in enumerate(labels1):
    print(n)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
